# Building / production-method I/O matrix (vanilla)

Uses **game definitions only**: `goods_data.vanilla_df`, vanilla buildings, and vanilla external production methods. Mod content is not applied.

Wide table: each row is a building and PM with registered good output. Columns `i_*` / `o_*` cover **all vanilla trade goods** (mostly zeros). Full frame is `df`; preview drops all-zero `i_*`/`o_*` columns.

In [ ]:
import pandas as pd

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData
from core.data.goods_data import GoodsData
from core.data.building_pm_io import build_pm_io_matrix
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config["game_path"], config["mod_path"])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)

goods_data.load_all()
building_data.load_all()

df = build_pm_io_matrix(building_data, goods_data, merged=False)
print(df.shape)
pd.options.display.max_columns = None
pd.options.display.precision = 4

In [ ]:
def sparse_io_preview(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop `i_*` and `o_*` columns that are all zero for a readable preview."""
    base = (
        ["building", "max_levels", "production_method"]
        if "max_levels" in frame.columns
        else ["building", "production_method"]
    )
    rest = [c for c in frame.columns if c not in base]
    nonzero = [c for c in rest if frame[c].abs().sum() > 0]
    return frame[base + sorted(nonzero, key=lambda x: (x[0], x))]


sparse_io_preview(df).head(20)